# Kaggle Pipeline - Train SASRec Baseline

Notebook nay train baseline `SASRec` tu bo du lieu da co:
- `item_metadata_final.jsonl`
- `sasrec_train_sequences.jsonl`
- `sasrec_valid_sequences.jsonl`
- `sasrec_test_sequences.jsonl`

## Input can co
- 1 dataset `ready-items` chua `item_metadata_final.jsonl`
- 1 dataset `mock-user-interactions` chua `mock_outputs/`
- Hoac file nam trong `/kaggle/working/`

## Output se tao ra
- `sasrec_outputs/train_history.json`
- `sasrec_outputs/valid_metrics.json`
- `sasrec_outputs/test_metrics.json`
- `sasrec_outputs/model_state.pt`

## Muc tieu
- Kiem tra nhanh pipeline train co chay end-to-end khong
- Tao baseline HitRate / NDCG cho phase recommender dau tien
- Co san notebook de thay mock data bang real data sau nay


In [ ]:
%pip install -q orjson


In [ ]:
from pathlib import Path
import math
import json
import random
import warnings
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd
import orjson
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True

READY_ITEMS_ROOT = Path("/kaggle/input/datasets/sonisson/ready-items/publish")
INTERACTIONS_ROOT = Path("/kaggle/input/datasets/sonisson/mock-user-interactions/mock_outputs")
WORK_DIR = Path("/kaggle/working")
OUTPUT_DIR = WORK_DIR / "sasrec_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ITEM_METADATA_CANDIDATES = [
    READY_ITEMS_ROOT / "ready_items_v1" / "item_metadata_final.jsonl",
    READY_ITEMS_ROOT / "item_metadata_final.jsonl",
    WORK_DIR / "publish" / "ready_items_v1" / "item_metadata_final.jsonl",
    WORK_DIR / "outputs" / "item_metadata_final.jsonl",
]
TRAIN_SEQ_CANDIDATES = [
    INTERACTIONS_ROOT / "sasrec_train_sequences.jsonl",
    WORK_DIR / "mock_outputs" / "sasrec_train_sequences.jsonl",
]
VALID_SEQ_CANDIDATES = [
    INTERACTIONS_ROOT / "sasrec_valid_sequences.jsonl",
    WORK_DIR / "mock_outputs" / "sasrec_valid_sequences.jsonl",
]
TEST_SEQ_CANDIDATES = [
    INTERACTIONS_ROOT / "sasrec_test_sequences.jsonl",
    WORK_DIR / "mock_outputs" / "sasrec_test_sequences.jsonl",
]

ITEM_METADATA_PATH = next((p for p in ITEM_METADATA_CANDIDATES if p.exists()), ITEM_METADATA_CANDIDATES[0])
TRAIN_SEQ_PATH = next((p for p in TRAIN_SEQ_CANDIDATES if p.exists()), TRAIN_SEQ_CANDIDATES[0])
VALID_SEQ_PATH = next((p for p in VALID_SEQ_CANDIDATES if p.exists()), VALID_SEQ_CANDIDATES[0])
TEST_SEQ_PATH = next((p for p in TEST_SEQ_CANDIDATES if p.exists()), TEST_SEQ_CANDIDATES[0])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_SEQ_LEN = 20
EMBED_DIM = 64
NUM_HEADS = 2
NUM_BLOCKS = 2
DROPOUT = 0.2
BATCH_SIZE = 256
EPOCHS = 5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
EVAL_NEGATIVES = 100

print("ITEM_METADATA_PATH:", ITEM_METADATA_PATH)
print("TRAIN_SEQ_PATH:", TRAIN_SEQ_PATH)
print("VALID_SEQ_PATH:", VALID_SEQ_PATH)
print("TEST_SEQ_PATH:", TEST_SEQ_PATH)
print("DEVICE:", DEVICE)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "rb") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(orjson.loads(line))
    return rows

def write_json(data: Dict[str, Any], path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        f.write(orjson.dumps(data, option=orjson.OPT_INDENT_2))

def pad_sequence(seq: List[int], max_len: int) -> List[int]:
    seq = seq[-max_len:]
    if len(seq) < max_len:
        seq = [0] * (max_len - len(seq)) + seq
    return seq

if not ITEM_METADATA_PATH.exists():
    raise FileNotFoundError(f"Khong tim thay item metadata: {ITEM_METADATA_PATH}")
if not TRAIN_SEQ_PATH.exists() or not VALID_SEQ_PATH.exists() or not TEST_SEQ_PATH.exists():
    raise FileNotFoundError("Khong tim thay file sequence train/valid/test")

item_rows = read_jsonl(ITEM_METADATA_PATH)
train_rows = read_jsonl(TRAIN_SEQ_PATH)
valid_rows = read_jsonl(VALID_SEQ_PATH)
test_rows = read_jsonl(TEST_SEQ_PATH)

num_items = 0
user_history: Dict[str, set] = {}
for rows in [train_rows, valid_rows, test_rows]:
    for row in rows:
        seq = [int(x) for x in row.get("input_item_ids", [])]
        tgt = int(row.get("target_item_id", 0))
        user_id = row.get("user_id", "")
        num_items = max(num_items, max(seq) if seq else 0, tgt)
        if user_id:
            user_history.setdefault(user_id, set()).update(seq)
            if tgt > 0:
                user_history[user_id].add(tgt)

dataset_summary = {
    "items_metadata": len(item_rows),
    "train_samples": len(train_rows),
    "valid_samples": len(valid_rows),
    "test_samples": len(test_rows),
    "num_items": int(num_items),
    "num_users": len(user_history),
}
print(json.dumps(dataset_summary, indent=2, ensure_ascii=False))
pd.DataFrame({"split": ["train", "valid", "test"], "rows": [len(train_rows), len(valid_rows), len(test_rows)]})


In [ ]:
class SASRecTrainDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]], num_items: int, max_seq_len: int):
        self.rows = rows
        self.num_items = num_items
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx: int):
        row = self.rows[idx]
        seq = [int(x) for x in row["input_item_ids"]]
        seq = pad_sequence(seq, self.max_seq_len)
        pos_item = int(row["target_item_id"])
        neg_item = random.randint(1, self.num_items)
        while neg_item == pos_item:
            neg_item = random.randint(1, self.num_items)
        return {
            "sequence": torch.tensor(seq, dtype=torch.long),
            "pos_item": torch.tensor(pos_item, dtype=torch.long),
            "neg_item": torch.tensor(neg_item, dtype=torch.long),
        }

class EvalDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]], num_items: int, max_seq_len: int):
        self.rows = rows
        self.num_items = num_items
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx: int):
        row = self.rows[idx]
        seq = [int(x) for x in row["input_item_ids"]]
        return {
            "user_id": row.get("user_id", ""),
            "sequence": torch.tensor(pad_sequence(seq, self.max_seq_len), dtype=torch.long),
            "target_item": int(row["target_item_id"]),
        }

class SASRecModel(nn.Module):
    def __init__(self, num_items: int, max_seq_len: int, embed_dim: int, num_heads: int, num_blocks: int, dropout: float):
        super().__init__()
        self.num_items = num_items
        self.max_seq_len = max_seq_len
        self.item_emb = nn.Embedding(num_items + 1, embed_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_seq_len, embed_dim)
        self.dropout = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_blocks)
        self.layer_norm = nn.LayerNorm(embed_dim)

    def encode(self, sequence: torch.Tensor) -> torch.Tensor:
        seq_len = sequence.size(1)
        pos_ids = torch.arange(seq_len, device=sequence.device).unsqueeze(0).expand_as(sequence)
        x = self.item_emb(sequence) + self.pos_emb(pos_ids)
        x = self.dropout(x)
        padding_mask = sequence.eq(0)
        causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=sequence.device), diagonal=1).bool()
        hidden = self.encoder(x, mask=causal_mask, src_key_padding_mask=padding_mask)
        hidden = self.layer_norm(hidden)
        lengths = sequence.ne(0).sum(dim=1).clamp(min=1) - 1
        batch_idx = torch.arange(sequence.size(0), device=sequence.device)
        return hidden[batch_idx, lengths]

    def score_items(self, seq_repr: torch.Tensor, item_ids: torch.Tensor) -> torch.Tensor:
        item_vec = self.item_emb(item_ids)
        return (seq_repr * item_vec).sum(dim=-1)

train_dataset = SASRecTrainDataset(train_rows, num_items=num_items, max_seq_len=MAX_SEQ_LEN)
valid_dataset = EvalDataset(valid_rows, num_items=num_items, max_seq_len=MAX_SEQ_LEN)
test_dataset = EvalDataset(test_rows, num_items=num_items, max_seq_len=MAX_SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = SASRecModel(num_items=num_items, max_seq_len=MAX_SEQ_LEN, embed_dim=EMBED_DIM, num_heads=NUM_HEADS, num_blocks=NUM_BLOCKS, dropout=DROPOUT).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()

print(model)


In [ ]:
def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer, criterion, device: str) -> float:
    model.train()
    losses = []
    for batch in tqdm(loader, desc="train", leave=False):
        sequence = batch["sequence"].to(device)
        pos_item = batch["pos_item"].to(device)
        neg_item = batch["neg_item"].to(device)

        seq_repr = model.encode(sequence)
        pos_logits = model.score_items(seq_repr, pos_item)
        neg_logits = model.score_items(seq_repr, neg_item)

        pos_labels = torch.ones_like(pos_logits)
        neg_labels = torch.zeros_like(neg_logits)
        loss = criterion(pos_logits, pos_labels) + criterion(neg_logits, neg_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(float(loss.item()))
    return float(np.mean(losses)) if losses else 0.0

def sample_negatives(user_id: str, target_item: int, num_items: int, history_map: Dict[str, set], num_negatives: int) -> List[int]:
    blocked = set(history_map.get(user_id, set()))
    blocked.add(target_item)

    pool = [item_id for item_id in range(1, num_items + 1) if item_id not in blocked]
    if not pool:
        pool = [item_id for item_id in range(1, num_items + 1) if item_id != target_item]
    if not pool:
        return [target_item]

    if len(pool) >= num_negatives:
        return random.sample(pool, num_negatives)
    return random.choices(pool, k=num_negatives)

def evaluate_topk(model: nn.Module, loader: DataLoader, num_items: int, history_map: Dict[str, set], device: str, k: int = 10, num_negatives: int = 100) -> Dict[str, float]:
    model.eval()
    hit_scores = []
    ndcg_scores = []

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"eval@{k}", leave=False):
            sequences = batch["sequence"].to(device)
            seq_repr = model.encode(sequences)

            candidate_rows = []
            user_ids = list(batch["user_id"])
            target_items = batch["target_item"].tolist() if hasattr(batch["target_item"], "tolist") else list(batch["target_item"])

            for user_id, target_item in zip(user_ids, target_items):
                negatives = sample_negatives(user_id, int(target_item), num_items, history_map, num_negatives)
                candidate_rows.append([int(target_item)] + [int(x) for x in negatives])

            candidate_tensor = torch.tensor(candidate_rows, dtype=torch.long, device=device)
            candidate_vectors = model.item_emb(candidate_tensor)
            logits = (candidate_vectors * seq_repr.unsqueeze(1)).sum(dim=-1)
            ranked_positions = torch.argsort(logits, dim=1, descending=True)
            target_ranks = (ranked_positions == 0).nonzero(as_tuple=False)[:, 1] + 1

            hit_batch = (target_ranks <= k).float().cpu().tolist()
            ndcg_batch = torch.where(
                target_ranks <= k,
                1.0 / torch.log2(target_ranks.float() + 1.0),
                torch.zeros_like(target_ranks, dtype=torch.float32),
            ).cpu().tolist()

            hit_scores.extend(hit_batch)
            ndcg_scores.extend(ndcg_batch)

    return {
        f"hit@{k}": round(float(np.mean(hit_scores)) if hit_scores else 0.0, 4),
        f"ndcg@{k}": round(float(np.mean(ndcg_scores)) if ndcg_scores else 0.0, 4),
    }

history = []
best_valid_hit = -1.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    valid_metrics = evaluate_topk(model, valid_loader, num_items, user_history, DEVICE, k=10, num_negatives=EVAL_NEGATIVES)
    row = {
        "epoch": epoch,
        "train_loss": round(train_loss, 4),
        **valid_metrics,
    }
    history.append(row)
    print(row)

    if valid_metrics["hit@10"] > best_valid_hit:
        best_valid_hit = valid_metrics["hit@10"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict(best_state)

valid_metrics = evaluate_topk(model, valid_loader, num_items, user_history, DEVICE, k=10, num_negatives=EVAL_NEGATIVES)
test_metrics = evaluate_topk(model, test_loader, num_items, user_history, DEVICE, k=10, num_negatives=EVAL_NEGATIVES)

history_path = OUTPUT_DIR / "train_history.json"
valid_metrics_path = OUTPUT_DIR / "valid_metrics.json"
test_metrics_path = OUTPUT_DIR / "test_metrics.json"
model_path = OUTPUT_DIR / "model_state.pt"

write_json({"history": history}, history_path)
write_json(valid_metrics, valid_metrics_path)
write_json(test_metrics, test_metrics_path)
torch.save(model.state_dict(), model_path)

print("saved:", history_path)
print("saved:", valid_metrics_path)
print("saved:", test_metrics_path)
print("saved:", model_path)
print("valid:", valid_metrics)
print("test:", test_metrics)
pd.DataFrame(history)
